[//]: # (cr:doc name='chapter_c04_batch_inference' id=01b14776)
# Chapter c04: Batch Inference (Causal Track)

Refreshes the `predictions` Delta table by scoring the current feature-store snapshot against the registered `@production` model. Independent from c02 (archetypes) and c05 (snapshot + dashboard) so operators can re-run scoring without recomputing archetypes.

`BATCH_INFERENCE_MODE='auto'` (default) scores only when `predictions` is missing or older than `PREDICTIONS_STALE_AFTER_HOURS`. Use `'always'` to force a fresh scoring run or `'never'` to skip (useful when inspecting the model without writing).


In [ ]:
# @cr:code name='init_progress' id=49f15248
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous

accept_workflow_params()
track_and_export_previous("c04_batch_inference.ipynb")
# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


[//]: # (cr:doc name='c04_configuration' id=e124bf24)
## Configuration

The cell below is the only place you should need to edit. Every value here is read by the batch-inference cell — nothing is hardcoded inside the algorithmic cells.

- **`BATCH_INFERENCE_MODE`** — `"auto"` (default): run only if `predictions` is missing or stale; `"always"`: force a scoring run regardless of freshness; `"never"`: skip entirely.
- **`PREDICTIONS_STALE_AFTER_HOURS`** — in `"auto"` mode, re-score if the latest `inference_point_in_time` in `predictions` is older than this many hours.
- **`SCORING_THRESHOLD`** — probability cutoff that marks a row as a predicted churner. Must match the threshold used in `decision_policy`.
- **`RISK_TIER_HIGH` / `RISK_TIER_MEDIUM`** — risk-tier cutoffs written onto each scored row; c05's snapshot writer reads these back when `SNAPSHOT_RISK_TIER_*` are `None`.


In [ ]:
# @cr:config name='configuration' id=99787c69
BATCH_INFERENCE_MODE = "auto"           # "auto" | "always" | "never"
PREDICTIONS_STALE_AFTER_HOURS = 24

SCORING_THRESHOLD = 0.5
RISK_TIER_HIGH = 0.6
RISK_TIER_MEDIUM = 0.3

# === Optional engagement overrides — leave None for auto-detection ====
# RunNamespace.resolve() honours these first, then falls back to
# discovery tiers (CR_RUN_ID env, project pointer .cr_active_run.json,
# experiments_root/runs/.active_run_id sentinel, latest-by-mtime).
# Pin them when several runs share an experiments dir and auto-detection
# picked the wrong one.
ENGAGEMENT_RUN_ID = None
ENGAGEMENT_EXPERIMENTS_DIR = None
MODEL_URI_OVERRIDE = None  # if None, looked up from MLflow @production alias


[//]: # (cr:doc name='c04_batch_inference_setup' id=e4fb8e8b)
## 4.0 Setup

Resolves catalog / schema / model identifiers from `ScoringConfig` (reads the persisted Databricks init JSON on Databricks, or the local pipeline's `best_model_meta.json` for local runs). The composite-name-qualified gold features table name is derived here so the algorithmic cells stay free of path-construction logic.


In [ ]:
# @cr:code name='setup_and_resolve_model' id=3c7ff9a0
from customer_retention.core.compat.detection import get_spark_session, is_databricks
from customer_retention.core.config import get_playbooks_dir
from customer_retention.stages.scoring import resolve_scoring_context

spark = get_spark_session()
PLAYBOOKS_DIR = get_playbooks_dir()

# Auto-detect the active run / model via RunNamespace's file-tracked
# discovery tiers (CR_RUN_ID env, project pointer .cr_active_run.json,
# experiments_root/runs/.active_run_id sentinel, latest-by-mtime). On
# Databricks, the model URI is resolved from MLflow's @production alias
# for the registered model named in training_metadata.json. Operator
# overrides from the configuration cell above (ENGAGEMENT_RUN_ID /
# ENGAGEMENT_EXPERIMENTS_DIR / MODEL_URI_OVERRIDE) short-circuit each
# resolution tier when set; default None → auto-detect.
_resolved = resolve_scoring_context(
    run_id=globals().get("ENGAGEMENT_RUN_ID"),
    experiments_dir=globals().get("ENGAGEMENT_EXPERIMENTS_DIR"),
    model_uri=globals().get("MODEL_URI_OVERRIDE"),
)
scoring_config = _resolved.scoring_config
_namespace = _resolved.namespace
_ns_source = _resolved.source
CATALOG = scoring_config.catalog if is_databricks() else "local"
SCHEMA = scoring_config.schema if is_databricks() else "local"
MODEL_NAME = _resolved.model_name
MODEL_VERSION = _resolved.model_version
MODEL_URI = _resolved.model_uri

COMPOSITE_NAME = scoring_config.composite_name
GOLD_FEATURES_FQN = (
    f"{CATALOG}.{SCHEMA}.gold_features_{COMPOSITE_NAME}"
    if COMPOSITE_NAME
    else f"{CATALOG}.{SCHEMA}.gold_features"
)

ARCHETYPE_CATALOG_FQN = f"{CATALOG}.{SCHEMA}.archetype_catalog"
ELIGIBILITY_POLICY_FQN = f"{CATALOG}.{SCHEMA}.eligibility_policy"
DECISION_POLICY_FQN = f"{CATALOG}.{SCHEMA}.decision_policy"
ELIGIBILITY_SNAPSHOT_FQN = f"{CATALOG}.{SCHEMA}.eligibility_snapshot"
PREDICTIONS_FQN = f"{CATALOG}.{SCHEMA}.predictions"
TOP_SHAP_DRIVERS_FQN = f"{CATALOG}.{SCHEMA}.top_shap_drivers"

print(f"Resolved playbooks_dir: {PLAYBOOKS_DIR}")
print(f"Active run namespace:   {_namespace.run_id if _namespace else '(none)'}")
print(f"Run source:             {_ns_source}")
print(f"Catalog/schema:         {CATALOG}.{SCHEMA}")
print(f"Composite name:         {COMPOSITE_NAME or '(unset)'}")
print(f"Gold features table:    {GOLD_FEATURES_FQN}")
print(f"Model URI:              {MODEL_URI or '(local)'}")
print(f"Model version:          {MODEL_VERSION}")


[//]: # (cr:doc name='c04_refresh_section' id=4a814e0b)
## 4.1 Refresh Predictions


In [ ]:
# @cr:code name='refresh_predictions' id=d5f90d2d
from datetime import datetime, timezone

from customer_retention.stages.scoring.batch_inference import (
    BatchInferenceConfig,
    run_batch_inference,
)


def _resolve_scope_filter():
    """Return ``(filter_expr, landing_table_fqn, target_name)`` for the target
    dataset's NB00 ``ProjectContext.sample_filters`` entry. The filter narrows
    the scoring population to the entity subset in force during exploration /
    training. ``landing_table_fqn`` is the source table the framework routes
    the filter through — the customer table (gold) has categoricals one-hot
    encoded, so a filter referencing a raw landing column like
    ``REVENUE_MARKET_SEGMENT`` only resolves against ``landing_<target>``.
    Returns ``(None, None, None)`` when project_context is absent / empty."""
    try:
        from customer_retention.analysis.auto_explorer.project_context import ProjectContext
        from customer_retention.analysis.auto_explorer.run_namespace import RunNamespace
    except ImportError:
        return None, None, None
    _ns = RunNamespace.from_env_or_latest()
    if _ns is None:
        return None, None, None
    _path = _ns.project_context_path
    if not _path.exists():
        return None, None, None
    _ctx = ProjectContext.load(_path)
    _filters = getattr(_ctx, "sample_filters", None) or {}
    if not _filters:
        return None, None, None
    _target_name = next(
        (name for name, ds in _ctx.datasets.items()
         if getattr(ds, "role", None) == "target"),
        None,
    )
    if _target_name is None and len(_ctx.datasets) == 1:
        _target_name = next(iter(_ctx.datasets))
    if _target_name is None:
        return None, None, None
    _filter_expr = _filters.get(_target_name)
    if not _filter_expr:
        return None, None, _target_name
    _landing_fqn = f"{CATALOG}.{SCHEMA}.landing_{_target_name}"
    return _filter_expr, _landing_fqn, _target_name


_scope_filter, _filter_via_table, _target_dataset_name = _resolve_scope_filter()

batch_inference_result = None
_predictions_status = "UNKNOWN"

if spark is None:
    _predictions_status = "SKIPPED: no Spark session (Databricks-only cell)"
    print(_predictions_status)
elif BATCH_INFERENCE_MODE == "never":
    _predictions_status = "SKIPPED: BATCH_INFERENCE_MODE='never'"
    print(_predictions_status)
else:
    _should_run = True
    _stale_reason = "always" if BATCH_INFERENCE_MODE == "always" else None
    if BATCH_INFERENCE_MODE == "auto":
        if not spark.catalog.tableExists(PREDICTIONS_FQN):
            _stale_reason = f"{PREDICTIONS_FQN} does not exist"
        else:
            _latest_ts_row = spark.sql(
                f"SELECT max(inference_point_in_time) AS ts FROM {PREDICTIONS_FQN}"
            ).head()
            _latest_ts = _latest_ts_row["ts"] if _latest_ts_row is not None else None
            if _latest_ts is None:
                _stale_reason = f"{PREDICTIONS_FQN} is empty"
            else:
                if _latest_ts.tzinfo is None:
                    _latest_ts = _latest_ts.replace(tzinfo=timezone.utc)
                _age_hours = (datetime.now(timezone.utc) - _latest_ts).total_seconds() / 3600.0
                if _age_hours > PREDICTIONS_STALE_AFTER_HOURS:
                    _stale_reason = f"latest inference_point_in_time is {_age_hours:.1f}h old (> {PREDICTIONS_STALE_AFTER_HOURS}h)"
                else:
                    _should_run = False
                    _predictions_status = (
                        f"FRESH: latest inference_point_in_time is {_age_hours:.1f}h old "
                        f"(<= {PREDICTIONS_STALE_AFTER_HOURS}h) — skipping"
                    )
                    print(_predictions_status)
    if _should_run:
        # When the scope filter is set, route it through `landing_<target>`
        # so raw categorical columns (one-hot encoded in gold) still resolve.
        # When `landing_<target>` does not exist (rare — only if landing was
        # never written), fall back to direct gold filter and let Spark
        # raise UNRESOLVED_COLUMN with the suggested one-hot alternatives.
        _resolved_filter_via_table = None
        if _scope_filter and _filter_via_table and spark.catalog.tableExists(_filter_via_table):
            _resolved_filter_via_table = _filter_via_table

        print(f"Running batch inference ({_stale_reason})")
        if _scope_filter:
            print(f"Scope filter (from NB00 project_context): {_scope_filter}")
            if _resolved_filter_via_table:
                print(f"  Routed via: {_resolved_filter_via_table} (entity_id inner-join with gold)")
            else:
                print(f"  Routed via: gold directly (landing_{_target_dataset_name or '?'} not found)")
        else:
            print("Scope filter: (none — scoring full entity population)")
        config = BatchInferenceConfig(
            catalog=CATALOG,
            schema=SCHEMA,
            model_uri=MODEL_URI,
            customer_table=GOLD_FEATURES_FQN,
            threshold=SCORING_THRESHOLD,
            risk_tier_high=RISK_TIER_HIGH,
            risk_tier_medium=RISK_TIER_MEDIUM,
            inference_timestamp=datetime.now(timezone.utc),
            filter_expression=_scope_filter,
            filter_via_table=_resolved_filter_via_table,
        )
        batch_inference_result = run_batch_inference(config)
        _predictions_status = batch_inference_result.summary()
        print(batch_inference_result.long_summary())


[//]: # (cr:doc name='c04_summary_section' id=0910a8d7)
## 4.2 Print Run Summary


In [ ]:
# @cr:code name='print_run_summary' id=429afe3b
if batch_inference_result is None:
    print(f"Predictions status: {_predictions_status}")
else:
    print(f"Inference id: {batch_inference_result.inference_id}")
    print(f"Inference timestamp: {batch_inference_result.inference_timestamp}")
    print(f"Scored: {batch_inference_result.total_scored:,}")
    print(f"Predicted churners: {batch_inference_result.predicted_churners:,}")
    print(f"Mean probability: {batch_inference_result.avg_probability:.4f}")
    print(f"Target table: {batch_inference_result.target_table_fqn}")


In [ ]:
# @cr:code name='release_stage_memory' id=2a6327db
from customer_retention.core.compat import release_stage_memory

release_stage_memory()
